# mini-vLLM on a free GPU (Colab T4)

This lights up the Phase 6 Triton kernel and captures real GPU numbers at **zero cost**.

**First:** `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`, then run the cells top to bottom.

> Clones the public repo. If you fork it, point the clone URL below at your copy.

In [ ]:
!nvidia-smi -L

In [ ]:
# Colab already ships CUDA torch + Triton; install only the rest, no-deps for the package.
!git clone -q https://github.com/vineetsista/minivllm.git
%cd minivllm
!pip install -q transformers tokenizers safetensors huggingface-hub accelerate \
    fastapi 'uvicorn[standard]' httpx pydantic rich pytest
!pip install -q -e . --no-deps
import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. The Triton kernel passes on GPU

On CPU this test *skips* (no CUDA). On the T4 it runs the fused RMSNorm kernel and checks it against the reference.

In [ ]:
!python -m pytest tests/test_kernels.py -v

## 2. Kernel speedup (the Phase 6 number)

In [ ]:
!python -m scripts.kernel_bench --rows 16384 --dim 1024

## 3. Decode throughput on GPU

The same benchmark from the README, now on CUDA — naive vs KV cache.

In [ ]:
!python -m scripts.benchmark --device cuda --max-new-tokens 64

## 4. (Optional) the full correctness suite on GPU

Proves the from-scratch engine matches HuggingFace token-for-token on CUDA too.

In [ ]:
!python -m pytest --runslow -q

Paste the kernel speedup and GPU tok/s into the README's results table — that's the headline GPU figure, earned for free.